In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import math
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA # Keep PCA import in case it's used elsewhere, but remove explicit calls
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE # For Stepwise Multilinear

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, VotingRegressor
from sklearn.tree import DecisionTreeRegressor


from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline # For model pipelines

import xgboost as xgb # For XGBoost
import catboost as cb # For CatBoost
import lightgbm as lgb # For LightGBM

import warnings
import joblib # For saving models
import shap # For model explainability
from datetime import datetime # For timestamps in filenames
import json # For saving results
warnings.filterwarnings("ignore")



/home/malloy/Desktop/workspace/Data-Science/Zindi-Competitions/amini soil prediction challenge/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initial dataset paths

path = '../dataset/'
train_path = os.path.join(path, 'Train.csv')
test_path = os.path.join(path, 'Test.csv')
gap_train_path = os.path.join(path, 'Gap_Train.csv')
gap_test_path = os.path.join(path, 'Gap_Test.csv')
sample_submission_path = os.path.join(path, 'SampleSubmission.csv')


train_df_original = pd.read_csv(train_path)
test_df_original = pd.read_csv(test_path)
train_gap_df = pd.read_csv(gap_train_path)
test_gap_df = pd.read_csv(gap_test_path)
sample_submission = pd.read_csv(sample_submission_path)

# Extract PID from original test_df once
test_pids = test_df_original['PID']

In [3]:
# Additional data paths - Make sure to replace these with your actual parquet files
path2 = '../processed_data/'
# Create path2 if it doesn't exist
os.makedirs(path2, exist_ok=True)

additional_data_sources_info = [
    {'name': 'landsat8', 'path': os.path.join(path2, 'processed_landsat8.parquet')},
    {'name': 'mcd43a4', 'path': os.path.join(path2, 'processed_modis_mcd43a4.parquet')},
    {'name': 'mod09ga', 'path': os.path.join(path2, 'processed_modis_mod09ga.parquet')},
    {'name': 'mod11a1', 'path': os.path.join(path2, 'processed_modis_mod11a1.parquet')},
    {'name': 'mod13q1', 'path': os.path.join(path2, 'processed_modis_mod13q1.parquet')},
    {'name': 'mod16a2', 'path': os.path.join(path2, 'processed_modis_mod16a2.parquet')},
    {'name': 'sentinel1', 'path': os.path.join(path2, 'processed_sentinel1.parquet')},
    {'name': 'sentinel2', 'path': os.path.join(path2, 'processed_sentinel2.parquet')}
]

# Load additional data using Polars and convert to Pandas, or create dummy files
loaded_additional_dfs = []
import polars as pl # Import polars here

for item in additional_data_sources_info:
    file_path = item['path']
    if not os.path.exists(file_path):
        print(f"Warning: Placeholder file '{os.path.basename(file_path)}' not found. Creating a dummy .parquet file.")
        # Create a dummy dataframe with some random data mimicking a few columns
        dummy_cols = ['PID', 'Date'] + [f'feature_{j}' for j in range(5)] + target_variables
        dummy_data = {col: np.random.rand(100) for col in dummy_cols if col not in ['PID', 'Date']}
        dummy_data['PID'] = np.random.randint(1000, 2000, 100)
        dummy_data['Date'] = pd.to_datetime(pd.date_range(start='2020-01-01', periods=100, freq='D'))
        
        dummy_pl_df = pl.DataFrame(dummy_data)
        dummy_pl_df.write_parquet(file_path)
        print(f"  Dummy file created: {file_path}")
    
    try:
        # Load with Polars and convert to Pandas
        df_pl = pl.read_parquet(file_path)
        loaded_additional_dfs.append(df_pl.to_pandas())
        print(f"Loaded {os.path.basename(file_path)} (Polars to Pandas). Shape: {loaded_additional_dfs[-1].shape}")
    except Exception as e:
        print(f"Error loading {os.path.basename(file_path)}: {e}")
        print("Skipping this additional dataset.")



Loaded processed_landsat8.parquet (Polars to Pandas). Shape: (2547305, 7)
Loaded processed_modis_mcd43a4.parquet (Polars to Pandas). Shape: (7767381, 12)
Loaded processed_modis_mod09ga.parquet (Polars to Pandas). Shape: (7486447, 16)
Loaded processed_modis_mod11a1.parquet (Polars to Pandas). Shape: (2465791, 7)
Loaded processed_modis_mod13q1.parquet (Polars to Pandas). Shape: (545563, 19)
Loaded processed_modis_mod16a2.parquet (Polars to Pandas). Shape: (935363, 3)
Loaded processed_sentinel1.parquet (Polars to Pandas). Shape: (2175275, 7)
Loaded processed_sentinel2.parquet (Polars to Pandas). Shape: (11768, 12)


In [4]:
# Define target variables
target_variables = ['N', 'P', 'K', 'Zn', 'S', 'Ca', 'Mg', 'B', 'Cu', 'Fe', 'Mn']
# Define the order of target variables for the submission file
submission_target_order = ['N', 'P', 'K', 'Ca', 'Mg', 'S', 'Fe', 'Mn', 'Zn', 'Cu', 'B']


In [5]:
# Model creation functions
def create_models():
    """Create dictionary of models with their hyperparameter grids"""
    models = {
        'XGBoost': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', MultiOutputRegressor(xgb.XGBRegressor(random_state=17, n_jobs=-1, eval_metric='rmse'))) # eval_metric for non-binary/multi-class
            ]),
            'params': {
                'regressor__estimator__n_estimators': [1000, 5000, 10000], # Adjusted n_estimators for tuning efficiency
                'regressor__estimator__max_depth': [3, 5, 7], # Reduced for faster tuning
                'regressor__estimator__learning_rate': [0.01, 0.1, 0.2],
                'regressor__estimator__subsample': [0.8, 0.9, 1.0]
            }
        },
        'CatBoost': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', MultiOutputRegressor(cb.CatBoostRegressor(random_state=17, verbose=False)))
            ]),
            'params': {
                'regressor__estimator__iterations': [1000, 5000, 10000], # Adjusted iterations
                'regressor__estimator__depth': [4, 6, 8], # Reduced for faster tuning
                'regressor__estimator__learning_rate': [0.01, 0.1, 0.2]
            }
        },
        'LightGBM': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', MultiOutputRegressor(lgb.LGBMRegressor(random_state=17, verbose=-1, n_jobs=-1)))
            ]),
            'params': {
                'regressor__estimator__n_estimators': [1000, 5000, 10000], # Adjusted n_estimators
                'regressor__estimator__max_depth': [3, 5, 7], # Reduced for faster tuning
                'regressor__estimator__learning_rate': [0.01, 0.1, 0.2],
                'regressor__estimator__num_leaves': [31, 50] # Reduced for faster tuning
            }
        },
        'RandomForest': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', MultiOutputRegressor(RandomForestRegressor(random_state=17, n_jobs=-1)))
            ]),
            'params': {
                'regressor__estimator__n_estimators': [50, 100, 200, 500], # Adjusted n_estimators
                'regressor__estimator__max_depth': [5, 10, None], # Adjusted max_depth
                'regressor__estimator__min_samples_split': [2, 5] # Reduced for faster tuning
            }
        },
        'AdaBoost': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', MultiOutputRegressor(AdaBoostRegressor(random_state=17)))
            ]),
            'params': {
                'regressor__estimator__n_estimators': [1000, 2000, 3000], # Adjusted n_estimators
                'regressor__estimator__learning_rate': [0.01, 0.1, 1.0]
            }
        },
        'DecisionTree': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', MultiOutputRegressor(DecisionTreeRegressor(random_state=17)))
            ]),
            'params': {
                'regressor__estimator__max_depth': [5, 10, 15], # Adjusted max_depth
                'regressor__estimator__min_samples_split': [2, 5],
                'regressor__estimator__min_samples_leaf': [1, 2]
            }
        }
    }
    return models

def create_pls_model():
    """Create PLS Regression model separately due to different interface and tuning"""
    return {
        'model': Pipeline([
            ('regressor', PLSRegression(n_components=2)) # Default n_components for initial pipeline
        ]),
        'params': {
            'regressor__n_components': [2, 5, 10, 20] # Limited range for tuning speed
        }
    }

def create_stepwise_model():
    """Create Stepwise Multilinear Regression using RFE within a MultiOutputRegressor pipeline"""
    # Note: RFE is a feature selector. Its n_features_to_select can be tuned, but
    # it's not a 'model' in the same sense for SHAP feature importance directly from its 'estimators_'.
    # Here, it's set to a fixed n_features_to_select for demonstration.
    return {
        'model': Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', MultiOutputRegressor(RFE(LinearRegression(), n_features_to_select=10)))
        ]),
        'params': {
            # No params for RFE's n_features_to_select in this grid, but could be added:
            # 'regressor__estimator__n_features_to_select': [5, 10, 15]
        }
    }



In [6]:
def get_processed_data_for_stage(
    stage_name, 
    df_train_raw, 
    df_test_raw, 
    target_variables,
):
    """
    Applies basic preprocessing steps for a given stage and returns processed data
    along with fitted transformers for inference. This function now fits all transformers
    on the current 'df_train_raw' for the given stage.
    """
    print(f"\n--- Processing Data for Stage: {stage_name} ---")
    
    train_df = df_train_raw.copy()
    test_df = df_test_raw.copy() # Test data is consistent across stages, only train grows.

    # Store fitted objects for inference for *this specific stage*
    stage_transformers = {
        'imputation_means': {},
        'high_vif_features': [],
        'feature_scaler': None, # The final scaler for all features
        'X_train_cols': [] # Final feature columns used for training
    }

    # 1. Handle Missing Values (fit on train_df, transform both)
    missing_values = [col for col in train_df.columns if train_df[col].isnull().any()]
    for col in missing_values:
        mean_val = train_df[col].mean()
        train_df[col].fillna(mean_val, inplace=True)
        # Only fill if column exists in test_df (newly added data might not have all columns)
        if col in test_df.columns:
            test_df[col].fillna(mean_val, inplace=True)
        stage_transformers['imputation_means'][col] = mean_val
    print(f"Imputed missing values for: {list(stage_transformers['imputation_means'].keys())}")

    # 2. Scale bio1 and bio7 (common preprocessing)
    if 'bio1' in train_df.columns:
        train_df['bio1'] /= 10
        if 'bio1' in test_df.columns:
            test_df['bio1'] /= 10
    if 'bio7' in train_df.columns:
        train_df['bio7'] /= 10
        if 'bio7' in test_df.columns:
            test_df['bio7'] /= 10
    print("Scaled 'bio1' and 'bio7'.")

    # 3. Drop xhp20 (common preprocessing)
    if 'xhp20' in train_df.columns:
        train_df.drop('xhp20', axis=1, inplace=True)
    if 'xhp20' in test_df.columns:
        test_df.drop('xhp20', axis=1, inplace=True)
    print("Dropped 'xhp20'.")

    # 4. Drop 'site' and 'PID' (from features - PID is handled separately for submission)
    if 'site' in train_df.columns:
        train_df.drop('site', axis=1, inplace=True)
    if 'site' in test_df.columns:
        test_df.drop('site', axis=1, inplace=True)
    
    if 'PID' in train_df.columns:
        train_df.drop('PID', axis=1, inplace=True)
    # PID is handled at the very start for test_df_original and test_pids,
    # so we don't drop it from test_df copy here, only ensure it's not a feature.

    print("Dropped 'site' and 'PID' (from features).")

    # --- No Feature Engineering or PCA in this version ---
    print("Skipping advanced feature engineering and PCA as per request.")

    # VIF Filtering
    # Only calculate VIF on numerical features that are not targets
    numerical_cols_for_vif = [col for col in train_df.columns if train_df[col].dtype in ['int64', 'float64'] and col not in target_variables]
    
    if len(numerical_cols_for_vif) > 1: # VIF needs at least 2 features
        X_vif = train_df[numerical_cols_for_vif]

        vif_df = pd.DataFrame()
        vif_df["features"] = X_vif.columns
        # Handle potential division by zero if a feature has zero variance
        vif_df["VIF Factor"] = [variance_inflation_factor(X_vif.values, i) if X_vif.iloc[:, i].var() != 0 else np.inf for i in range(X_vif.shape[1])]
        
        high_vif_features = vif_df[vif_df['VIF Factor'] > 100]['features'].tolist()
        
        train_df.drop(columns=high_vif_features, errors='ignore', inplace=True)
        test_df.drop(columns=high_vif_features, errors='ignore', inplace=True)
        stage_transformers['high_vif_features'] = high_vif_features
        print(f"Dropped high VIF features: {high_vif_features}")
    else:
        print("Skipping VIF filtering: Not enough numerical features.")

    # Separate features (X) and targets (y)
    X_train_stage = train_df.drop(columns=target_variables, errors='ignore')
    y_train_stage = train_df[target_variables]

    # Ensure test_df still contains PID and align with train_df features later
    test_df_features_only = test_df.drop(columns=target_variables, errors='ignore') 
    
    # Align train and test dataframes to have the same columns and order for scaling and prediction
    X_train_cols_final = X_train_stage.columns.tolist()
    stage_transformers['X_train_cols'] = X_train_cols_final

    # Add missing columns to test_df_features_only (e.g., new features from additional datasets)
    missing_in_test_after_alignment = set(X_train_cols_final) - set(test_df_features_only.columns)
    for col in missing_in_test_after_alignment:
        test_df_features_only[col] = 0 # Fill missing columns in test with 0 (or appropriate value)

    # Filter and reorder test columns to match training columns
    X_test_stage = test_df_features_only[X_train_cols_final]
    
    # Final Scaling of all features (fits on train_stage, transforms both train/test_stage)
    numerical_cols_to_scale = [col for col in X_train_stage.columns if X_train_stage[col].dtype in ['int64', 'float64']]
    
    feature_scaler = StandardScaler()
    # Check if there are columns to scale before attempting fit_transform
    if numerical_cols_to_scale:
        X_train_stage[numerical_cols_to_scale] = feature_scaler.fit_transform(X_train_stage[numerical_cols_to_scale])
        X_test_stage[numerical_cols_to_scale] = feature_scaler.transform(X_test_stage[numerical_cols_to_scale])
    else:
        print("No numerical columns found to scale in this stage.")

    stage_transformers['feature_scaler'] = feature_scaler
    stage_transformers['scaled_feature_columns'] = numerical_cols_to_scale 
    print(f"Final scaling applied. Train shape: {X_train_stage.shape}, Test shape: {X_test_stage.shape}")

    return X_train_stage, X_test_stage, y_train_stage, stage_transformers


In [7]:
def tune_and_train_multioutput_model(model_name, model_config, X_train, y_train_multi, cv_folds=5, n_iter=20):
    """
    Performs RandomizedSearchCV for a multi-output model and returns the best estimator.
    Scoring is based on negative Root Mean Squared Error (neg_root_mean_squared_error).
    """
    print(f"  Tuning and training {model_name}...")
    
    pipeline = model_config['model']
    param_grid = model_config['params']
    
    # Handle PLSRegression specially for n_components validity
    if model_name == 'PLSRegression':
        # PLSRegression needs n_components <= min(n_features, n_samples)
        max_components_available = min(X_train.shape[1], X_train.shape[0])
        # Filter n_components in param_grid to be valid
        valid_n_components = [n for n in param_grid.get('regressor__n_components', []) if n <= max_components_available and n > 0]
        if not valid_n_components:
            print(f"    Warning: No valid n_components for PLSRegression given data shape ({max_components_available} max). Skipping tuning.")
            # Return a default PLS model if tuning is skipped
            if max_components_available > 0:
                default_n_comp = min(2, max_components_available)
                model = PLSRegression(n_components=default_n_comp)
                model.fit(X_train, y_train_multi)
                # Use neg_root_mean_squared_error for consistency
                cv_scores_default = cross_val_score(model, X_train, y_train_multi, cv=cv_folds, scoring='neg_root_mean_squared_error', n_jobs=-1)
                return model, cv_scores_default.mean()
            return None, None
        param_grid['regressor__n_components'] = valid_n_components
    
    try:
        # Use RandomizedSearchCV with neg_root_mean_squared_error as scoring
        random_search = RandomizedSearchCV(
            pipeline,
            param_grid,
            n_iter=n_iter,
            cv=cv_folds,
            scoring='neg_root_mean_squared_error', # Changed scoring to RMSE
            n_jobs=-1,
            random_state=17,
            verbose=0 # Suppress verbose output during search
        )
        random_search.fit(X_train, y_train_multi)
        print(f"    Best params for {model_name}: {random_search.best_params_}")
        return random_search.best_estimator_, random_search.best_score_
    except Exception as e:
        print(f"    Error during tuning {model_name}: {e}")
        return None, None


def evaluate_model(model, X_val, y_val, target_columns):
    """Evaluate model performance"""
    y_pred = model.predict(X_val)
    
    mae = mean_absolute_error(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, y_pred, multioutput='uniform_average') # Average R2 across targets
    
    # Per-target metrics
    per_target_metrics = {}
    for i, target in enumerate(target_columns):
        per_target_metrics[target] = {
            'mae': mean_absolute_error(y_val.iloc[:, i], y_pred[:, i]), # Use iloc for y_val
            'mse': mean_squared_error(y_val.iloc[:, i], y_pred[:, i]),
            'r2': r2_score(y_val.iloc[:, i], y_pred[:, i])
        }
    
    return {
        'overall_mae': mae,
        'overall_mse': mse,
        'overall_rmse': rmse,
        'overall_r2': r2,
        'per_target': per_target_metrics
    }


def train_and_evaluate_multioutput_models(X_train, X_test, y_train, y_test, target_variables, cv_folds=5, n_iter=20):
    """
    Train, tune, and evaluate multi-output regression models.
    Returns results, predictions, and the best performing model.
    The best model is determined by maximizing negative Root Mean Squared Error (RMSE).
    """
    results = {}
    predictions = {}
    # Initialize best_overall_model_info with a very low (highly negative) CV score for RMSE
    best_overall_model_info = {'model_name': 'None', 'cv_score': -np.inf, 'model': None}
    
    print("\nTraining and tuning multi-output regression models...")
    available_targets = [target for target in target_variables if target in y_train.columns]
    y_train_multi = y_train[available_targets]
    y_test_multi = y_test[available_targets] if y_test is not None else None
    
    # Get model configurations
    models_config = create_models()
    models_config['PLSRegression'] = create_pls_model()
    models_config['StepwiseMultilinear'] = create_stepwise_model() # Stepwise model

    trained_models_for_ensemble = [] # To store (model_name, fitted_model, cv_score) for ensembling
    
    for model_name, model_config in models_config.items():
        tuned_model, best_cv_score = None, None
        
        # Check if the model has a param_grid for tuning
        if model_config['params']:
            tuned_model, best_cv_score = tune_and_train_multioutput_model(
                model_name, model_config, X_train, y_train_multi, cv_folds=5, n_iter=n_iter
            )
        else: # For models with no tuning parameters (e.g., base LinearRegression if not RFE based, or fixed RFE)
            print(f"  Training {model_name} (no tuning specified or required)...")
            try:
                current_model = model_config['model']
                current_model.fit(X_train, y_train_multi)
                tuned_model = current_model
                
                # Calculate CV score for models not tuned via RandomizedSearchCV
                # Use neg_root_mean_squared_error for consistency
                cv_scores = cross_val_score(tuned_model, X_train, y_train_multi, cv=cv_folds, scoring='neg_root_mean_squared_error', n_jobs=-1)
                best_cv_score = cv_scores.mean()
            except Exception as e:
                print(f"    Error training {model_name} without tuning: {e}")
                tuned_model = None

        if tuned_model:
            # Store model for potential ensemble and track best overall
            trained_models_for_ensemble.append((model_name, tuned_model, best_cv_score))
            
            # Evaluate the best estimator from tuning (or directly trained model)
            try:
                y_train_pred = tuned_model.predict(X_train)
                y_test_pred = tuned_model.predict(X_test) if X_test is not None else None
                
                predictions[model_name] = {
                    'train_pred': pd.DataFrame(y_train_pred, columns=available_targets, index=y_train_multi.index),
                    'test_pred': pd.DataFrame(y_test_pred, columns=available_targets, index=y_test.index) if y_test_pred is not None else None
                }
                
                target_metrics = {}
                overall_train_r2 = [] # Still calculating R2 for reporting, but not for 'best' selection
                overall_test_r2 = []
                
                for i, target in enumerate(available_targets):
                    train_r2 = r2_score(y_train_multi[target], y_train_pred[:, i])
                    train_rmse = np.sqrt(mean_squared_error(y_train_multi[target], y_train_pred[:, i]))
                    train_mae = mean_absolute_error(y_train_multi[target], y_train_pred[:, i])
                    overall_train_r2.append(train_r2)
                    
                    test_r2, test_rmse, test_mae = None, None, None
                    if y_test_multi is not None:
                        test_r2 = r2_score(y_test_multi[target], y_test_pred[:, i])
                        test_rmse = np.sqrt(mean_squared_error(y_test_multi[target], y_test_pred[:, i]))
                        test_mae = mean_absolute_error(y_test_multi[target], y_test_pred[:, i])
                        overall_test_r2.append(test_r2)
                    
                    target_metrics[target] = {
                        'train_r2': train_r2, 'train_rmse': train_rmse, 'train_mae': train_mae,
                        'test_r2': test_r2, 'test_rmse': test_rmse, 'test_mae': test_mae
                    }
                
                results[model_name] = {
                    'cv_r2_mean': best_cv_score, # This is actually negative RMSE, renamed for consistency
                    'cv_r2_std': np.std(cv_scores) if 'cv_scores' in locals() and cv_scores is not None else np.nan,
                    'overall_train_r2': np.mean(overall_train_r2),
                    'overall_test_r2': np.mean(overall_test_r2) if overall_test_r2 else None,
                    'target_metrics': target_metrics,
                    'model': tuned_model # Store the full pipeline/model
                }
                
                # Compare based on negative RMSE (higher negative RMSE is better, i.e., closer to 0)
                if best_cv_score > best_overall_model_info['cv_score']:
                    best_overall_model_info = {
                        'model_name': model_name,
                        'cv_score': best_cv_score, # Negative RMSE
                        'model': tuned_model
                    }
                
            except Exception as e:
                print(f"  Error during evaluation of {model_name}: {e}")
        else:
            print(f"  {model_name} could not be trained or tuned successfully.")

    # --- Ensemble with VotingRegressor (Top 5 Models) ---
    print("\nAttempting to create VotingRegressor Ensemble...")
    
    # Sort models by their CV score (negative RMSE) in descending order
    # A higher negative RMSE (closer to zero) is better.
    ensemble_candidates = sorted(trained_models_for_ensemble, key=lambda x: x[2] if x[2] is not None else -np.inf, reverse=True)
    
    top_5_estimators_for_voting = []
    
    for i, (model_name, model_pipeline, cv_score) in enumerate(ensemble_candidates):
        if i >= 5: # Take top 5
            break
        
        # Check if the model is a Pipeline and has a 'regressor' step
        if isinstance(model_pipeline, Pipeline) and 'regressor' in model_pipeline.named_steps:
            # Get the actual regressor from the MultiOutputRegressor (which is a Pipeline step)
            multi_output_regressor = model_pipeline.named_steps['regressor']
            if isinstance(multi_output_regressor, MultiOutputRegressor):
                base_estimator = multi_output_regressor.estimator # This is the single-output regressor
                # Add the base estimator (unwrapped) for VotingRegressor
                top_5_estimators_for_voting.append((model_name, base_estimator))
            else:
                # If it's not MultiOutputRegressor (e.g., PLSRegression wrapped directly in Pipeline),
                # it cannot be directly used as a base estimator for a MultiOutputRegressor(VotingRegressor).
                # We handle PLSRegression as a separate model.
                print(f"  Skipping {model_name} for VotingRegressor: Not a standard MultiOutputRegressor wrapped model.")
        elif isinstance(model_pipeline, PLSRegression):
            print(f"  Skipping {model_name} for VotingRegressor: PLSRegression is inherently multi-output and not directly compatible as a base estimator for MultiOutputRegressor(VotingRegressor).")
        else:
            print(f"  Skipping {model_name} for VotingRegressor: Not a recognized pipeline structure or model type.")
            
    
    if len(top_5_estimators_for_voting) >= 2: # Need at least 2 suitable estimators for VotingRegressor
        try:
            print(f"  Ensembling with: {[name for name, _ in top_5_estimators_for_voting]}")
            # The VotingRegressor itself should be wrapped by MultiOutputRegressor
            voting_regressor = VotingRegressor(estimators=top_5_estimators_for_voting, n_jobs=-1)
            multioutput_voting_model = MultiOutputRegressor(voting_regressor)

            # Use neg_root_mean_squared_error for consistency
            cv_scores_ensemble = cross_val_score(multioutput_voting_model, X_train, y_train_multi, cv=cv_folds, scoring='neg_root_mean_squared_error', n_jobs=-1)
            multioutput_voting_model.fit(X_train, y_train_multi)

            y_train_pred_ensemble = multioutput_voting_model.predict(X_train)
            y_test_pred_ensemble = multioutput_voting_model.predict(X_test) if X_test is not None else None
            
            overall_train_r2_ensemble = []
            overall_test_r2_ensemble = []
            target_metrics_ensemble = {}

            for i, target in enumerate(available_targets):
                train_r2_e = r2_score(y_train_multi[target], y_train_pred_ensemble[:, i])
                overall_train_r2_ensemble.append(train_r2_e)

                test_r2_e = None
                if y_test_multi is not None:
                    test_r2_e = r2_score(y_test_multi[target], y_test_pred_ensemble[:, i])
                    overall_test_r2_ensemble.append(test_r2_e)
                
                target_metrics_ensemble[target] = {'train_r2': train_r2_e, 'test_r2': test_r2_e}

            ensemble_name = "VotingRegressor (Top 5)"
            results[ensemble_name] = {
                'cv_r2_mean': cv_scores_ensemble.mean(), # Negative RMSE
                'cv_r2_std': cv_scores_ensemble.std(),
                'overall_train_r2': np.mean(overall_train_r2_ensemble),
                'overall_test_r2': np.mean(overall_test_r2_ensemble) if overall_test_r2_ensemble else None,
                'target_metrics': target_metrics_ensemble,
                'model': multioutput_voting_model
            }
            predictions[ensemble_name] = {
                'train_pred': pd.DataFrame(y_train_pred_ensemble, columns=available_targets, index=y_train_multi.index),
                'test_pred': pd.DataFrame(y_test_pred_ensemble, columns=available_targets, index=y_test.index) if y_test_pred_ensemble is not None else None
            }

            # Compare based on negative RMSE
            if cv_scores_ensemble.mean() > best_overall_model_info['cv_score']:
                best_overall_model_info = {
                    'model_name': ensemble_name,
                    'cv_score': cv_scores_ensemble.mean(), # Negative RMSE
                    'model': multioutput_voting_model
                }
            print(f"  VotingRegressor Ensemble trained. CV RMSE: {-cv_scores_ensemble.mean():.4f}") # Print positive RMSE
        except Exception as e:
            print(f"  Error training VotingRegressor Ensemble: {e}")
    else:
        print("  Not enough suitable trained models (at least 2 single-output-wrapped) to form a VotingRegressor ensemble.")

    return results, predictions, best_overall_model_info


In [8]:
# --- SHAP Explainability Function ---

def explain_model_with_shap(model, X_data, feature_names, plot_title="SHAP Summary Plot"):
    """
    Generates SHAP summary plot for a given model and data.
    Handles Pipeline and MultiOutputRegressor structures.
    """
    print(f"\n--- Generating SHAP Explanation for '{plot_title.replace('SHAP Summary Plot for ', '')}' ---")
    try:
        # Step 1: Extract the actual estimator to be explained and transformed data
        actual_estimator = None
        X_transformed_for_shap = X_data.copy()

        if isinstance(model, Pipeline):
            # If the pipeline contains a scaler, apply it
            if 'scaler' in model.named_steps:
                X_transformed_for_shap = model.named_steps['scaler'].transform(X_data)
            
            # Get the final regressor step
            if 'regressor' in model.named_steps:
                actual_estimator = model.named_steps['regressor']
        elif isinstance(model, (MultiOutputRegressor, PLSRegression)):
            actual_estimator = model
        
        if actual_estimator is None:
            print("  Could not identify a suitable estimator for SHAP explanation.")
            return

        # Handle MultiOutputRegressor wrapper: Explain the first base estimator
        if isinstance(actual_estimator, MultiOutputRegressor):
            if hasattr(actual_estimator, 'estimators_') and actual_estimator.estimators_:
                # If already fitted, use the fitted estimators
                base_estimator = actual_estimator.estimators_[0]
            elif hasattr(actual_estimator, 'estimator'):
                # If not yet fitted, use the base estimator directly
                base_estimator = actual_estimator.estimator
            else:
                print("  Could not find a base estimator within MultiOutputRegressor for SHAP.")
                return
            
            # If the base estimator is a VotingRegressor, pick one of its estimators (e.g., first)
            if isinstance(base_estimator, VotingRegressor):
                if base_estimator.estimators_:
                    print("  Explaining the first estimator within VotingRegressor for SHAP.")
                    base_estimator = base_estimator.estimators_[0]
                else:
                    print("  VotingRegressor has no base estimators. Skipping SHAP.")
                    return
            
            print(f"  Explaining wrapped estimator: {type(base_estimator).__name__}")
        else: # For models like PLSRegression that are directly multi-output
            base_estimator = actual_estimator
            print(f"  Explaining direct multi-output estimator: {type(base_estimator).__name__}")

        # Step 2: Sample data for SHAP (to avoid memory issues)
        sample_size = min(X_transformed_for_shap.shape[0], 500) # Max 500 samples
        X_shap_sample = X_transformed_for_shap.sample(n=sample_size, random_state=17)

        # Step 3: Create SHAP explainer
        explainer = None
        if hasattr(base_estimator, 'predict_proba') or hasattr(base_estimator, 'feature_importances_'):
            # Use TreeExplainer for tree-based models if applicable
            try:
                explainer = shap.TreeExplainer(base_estimator)
            except Exception:
                # Fallback if TreeExplainer fails (e.g., model not fully tree-like or bad input)
                print("  TreeExplainer failed, falling back to KernelExplainer (can be slow).")
        
        if explainer is None: # Use KernelExplainer for non-tree models or if TreeExplainer fails
            # Select a small, representative subset of X_data for KernelExplainer's background data
            background_sample_size = min(X_transformed_for_shap.shape[0], 50)
            background_data = shap.kmeans(X_transformed_for_shap, background_sample_size).data if X_transformed_for_shap.shape[0] > 0 else X_transformed_for_shap
            
            # KernelExplainer needs a predict function that takes X and returns single output or array for multi-output
            if isinstance(actual_estimator, MultiOutputRegressor):
                # For MultiOutputRegressor, model.predict returns (N_samples, N_outputs)
                explainer = shap.KernelExplainer(actual_estimator.predict, background_data)
            elif isinstance(actual_estimator, PLSRegression):
                # For PLSRegression, model.predict also returns (N_samples, N_outputs)
                explainer = shap.KernelExplainer(actual_estimator.predict, background_data)
            else:
                # Fallback for unexpected cases, or single-output models
                explainer = shap.KernelExplainer(actual_estimator.predict, background_data)
            
            print(f"  Using KernelExplainer with background data shape: {background_data.shape}")

        if explainer is None:
            print("  Could not create a SHAP explainer. Skipping explanation.")
            return {
                'feature_importance': {},
                'top_5_features': [],
                'shap_available': False,
                'error': "Could not create SHAP explainer."
            }

        shap_values = explainer.shap_values(X_shap_sample)

        # For multi-output, shap_values will be a list of arrays (one per output).
        # For summary plot, we often use the mean absolute SHAP value across all outputs.
        if isinstance(shap_values, list) and len(shap_values) > 1: # Multi-output case with multiple shap value arrays
            # Average the absolute SHAP values across all outputs for a single summary plot
            # Ensure shapes match X_shap_sample columns if possible
            if all(sv.shape[1] == X_shap_sample.shape[1] for sv in shap_values):
                avg_abs_shap_values = np.mean(np.abs(np.array(shap_values)), axis=0)
            else:
                # Fallback if shap_values shape is unexpected (e.g., for PLSRegression, it might return a single array)
                print("  Warning: SHAP values structure unexpected for multi-output average. Plotting first output.")
                avg_abs_shap_values = shap_values[0]
                
            shap.summary_plot(avg_abs_shap_values, X_shap_sample, feature_names=feature_names, show=False)
        else: # Single-output case (or PLSRegression might give a single array)
            if isinstance(shap_values, list) and len(shap_values) == 1: # List with one item
                shap.summary_plot(shap_values[0], X_shap_sample, feature_names=feature_names, show=False)
            else: # Directly an array
                shap.summary_plot(shap_values, X_shap_sample, feature_names=feature_names, show=False)
        
        plt.title(plot_title)
        plt.tight_layout()
        plt.show()

        # Calculate feature importance for return value
        importance = None
        if isinstance(shap_values, list):
            # Sum absolute SHAP values across outputs, then mean across samples
            if all(sv.shape[1] == X_shap_sample.shape[1] for sv in shap_values):
                importance = np.mean(np.sum(np.abs(np.array(shap_values)), axis=0), axis=0) # sum abs across outputs, mean across samples
            elif len(shap_values) == 1:
                 importance = np.mean(np.abs(shap_values[0]), axis=0)
            else:
                importance = None # Cannot calculate robustly
        else: # Single array
            importance = np.mean(np.abs(shap_values), axis=0)

        feature_importance_dict = {}
        if importance is not None and len(importance) == len(feature_names):
            feature_importance_dict = dict(zip(feature_names, importance))
            sorted_features = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)
            feature_importance_dict = dict(sorted_features)
        else:
            print("  Could not reliably calculate feature importances from SHAP values.")

        return {
            'feature_importance': feature_importance_dict,
            'top_5_features': list(feature_importance_dict.keys())[:5],
            'shap_available': True
        }
        
    except Exception as e:
        print(f"  Error generating SHAP explanation: {e}")
        print("  SHAP explanation skipped for this model/stage.")
        return {
            'feature_importance': {},
            'top_5_features': [],
            'shap_available': False,
            'error': str(e)
        }



In [9]:
overall_metrics_results = []
overall_submission_dfs = {}
stage_summaries = []

best_overall_model_across_stages = {'model_name': 'None', 'cv_score': -np.inf, 'model': None, 'stage': 'N/A'}

current_train_df_for_stages = train_df_original.copy()

# Stages: Original data + each additional dataset
evaluation_stages_names = ["Original Data"] + [f"Added Data {i+1} ({additional_data_sources_info[i]['name']})" for i in range(len(loaded_additional_dfs))]
# evaluation_data_sources will be `loaded_additional_dfs`

for i, stage_name in enumerate(evaluation_stages_names):
    print(f"\n======== Starting Evaluation for Stage: {stage_name} ========")
    
    if i > 0: # For "Added Data X" stages
        # Concatenate the current base training data with the new additional dataset
        new_data_df = loaded_additional_dfs[i-1].copy() # Get the i-1th additional dataset
        
        # Ensure new_data_df has 'PID' and target columns removed if they exist, to avoid conflict
        # Only concatenate features for training
        new_data_features_only = new_data_df.drop(columns=['PID'] + target_variables, errors='ignore')
        
        current_train_df_for_stages = pd.concat([current_train_df_for_stages, new_data_features_only], ignore_index=True)
        print(f"Dataset size after adding {additional_data_sources_info[i-1]['name']}: {current_train_df_for_stages.shape}")

    # Get preprocessed data for the current stage
    # All preprocessing steps are applied on this current_train_df_for_stages
    X_train_processed, X_test_processed_inference, y_train_processed, stage_transformers = get_processed_data_for_stage(
        stage_name, current_train_df_for_stages, test_df_original, target_variables
    )

    # Use a train/test split of the *current stage's training data* for model evaluation
    X_train_eval, X_test_eval, y_train_eval, y_test_eval = train_test_split(
        X_train_processed, y_train_processed, test_size=0.2, random_state=17
    )

    # --- Train and Evaluate Multi-Output Models (including tuning) ---
    stage_multi_results, stage_multi_predictions, stage_best_multi_model_info = train_and_evaluate_multioutput_models(
        X_train=X_train_eval,
        X_test=X_test_eval,
        y_train=y_train_eval,
        y_test=y_test_eval,
        target_variables=target_variables,
        cv_folds=5, # Cross-validation folds for evaluation
        n_iter=20 # Iterations for RandomizedSearchCV
    )

    # Compile multi-output model metrics for this stage
    stage_best_cv_r2 = -np.inf # This will store negative RMSE
    stage_best_model_name = "None"

    for model_name, metrics in stage_multi_results.items():
        if metrics['model'] is not None:
            # Add overall multi-output metrics
            row_overall = {
                'Stage': stage_name,
                'Model Type': 'MultiOutput',
                'Model Name': model_name,
                'Target': 'Overall',
                'CV RMSE Mean': -metrics['cv_r2_mean'], # Convert negative RMSE to positive for reporting
                'CV RMSE Std': metrics['cv_r2_std'],
                'Train R2': metrics['overall_train_r2'],
                'Train RMSE': np.nan, # Not directly computed for overall for now
                'Train MAE': np.nan,   # Not directly computed for overall for now
                'Test R2': metrics['overall_test_r2'],
                'Test RMSE': np.nan,   # Not directly computed for overall for now
                'Test MAE': np.nan     # Not directly computed for overall for now
            }
            overall_metrics_results.append(row_overall)
            
            # Add per-target metrics for multi-output models
            for target, target_metrics in metrics['target_metrics'].items():
                row_per_target = {
                    'Stage': stage_name,
                    'Model Type': 'MultiOutput',
                    'Model Name': model_name,
                    'Target': target,
                    'CV RMSE Mean': np.nan, # CV RMSE mean is overall for multi-output
                    'CV RMSE Std': np.nan,
                    'Train R2': target_metrics['train_r2'],
                    'Train RMSE': target_metrics['train_rmse'],
                    'Train MAE': target_metrics['train_mae'],
                    'Test R2': target_metrics['test_r2'],
                    'Test RMSE': target_metrics['test_rmse'],
                    'Test MAE': target_metrics['test_mae']
                }
                overall_metrics_results.append(row_per_target)

            # Compare based on negative RMSE (higher negative RMSE is better, i.e., closer to 0)
            if metrics['cv_r2_mean'] > stage_best_cv_r2:
                stage_best_cv_r2 = metrics['cv_r2_mean']
                stage_best_model_name = model_name
                # Update best model for this stage
                stage_best_multi_model_info = {'model_name': model_name, 'cv_score': metrics['cv_r2_mean'], 'model': metrics['model']}

    # Update overall best model across all stages
    if stage_best_multi_model_info['model'] is not None and stage_best_multi_model_info['cv_score'] > best_overall_model_across_stages['cv_score']:
        best_overall_model_across_stages['model_name'] = stage_best_multi_model_info['model_name']
        best_overall_model_across_stages['cv_score'] = stage_best_multi_model_info['cv_score']
        best_overall_model_across_stages['model'] = stage_best_multi_model_info['model']
        best_overall_model_across_stages['stage'] = stage_name
        # Print positive RMSE for readability
        print(f"New overall best model: {best_overall_model_across_stages['model_name']} from Stage '{stage_name}' (CV RMSE: {-best_overall_model_across_stages['cv_score']:.4f})")

    # --- Generate Predictions for Final Test Set (X_test_processed_inference) ---
    print(f"\n--- Generating final predictions for Stage: {stage_name} ---")

    # Multi-Output Model Final Predictions (using best overall multi-output model for THIS stage)
    multioutput_predictions_for_submission = pd.DataFrame({'PID': test_pids})
    if stage_best_multi_model_info['model'] is not None:
        preds = stage_best_multi_model_info['model'].predict(X_test_processed_inference)
        # Convert numpy array predictions to DataFrame, preserving index
        preds_df = pd.DataFrame(preds, columns=target_variables, index=X_test_processed_inference.index)
        
        # Merge with PID from original test_df to ensure correct alignment
        # This assumes PID is aligned with the original test_df.index for simplicity.
        # If test_df_original was reindexed during preprocessing, this merge needs adjustment.
        # Given how test_pids is extracted, it should align.
        
        # Create submission dataframe with required order
        for col in submission_target_order:
            multioutput_predictions_for_submission[col] = preds_df[col].values
        
        multioutput_predictions_for_submission = multioutput_predictions_for_submission[['PID'] + submission_target_order]
    else:
        for col in submission_target_order:
            multioutput_predictions_for_submission[col] = np.nan

    overall_submission_dfs[f'multioutput_model_submission_{stage_name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")}'] = multioutput_predictions_for_submission
    print(f"Generated multi-output model submission for '{stage_name}'.")

    # --- Model Explainability with SHAP (for the best model of this stage) ---
    if stage_best_multi_model_info['model'] is not None:
        explain_model_with_shap(
            stage_best_multi_model_info['model'],
            X_train_eval, # Use training eval data for SHAP explanation
            X_train_eval.columns.tolist(), # Feature names for SHAP plot
            plot_title=f"SHAP Summary Plot for {stage_best_multi_model_info['model_name']} (Stage: {stage_name})"
        )
    else:
        print(f"Skipping SHAP explanation for Stage '{stage_name}': No best model found.")


    # --- Compile Stage Summary ---
    current_stage_summary = (
        f"Stage: {stage_name}\n"
        f"  Current training data shape: {current_train_df_for_stages.shape}\n"
        f"  Best Multi-Output Model (this stage): {stage_best_model_name} (CV RMSE Mean: {-stage_best_cv_r2:.4f})\n" # Print positive RMSE
        f"  Overall Best Model (so far): {best_overall_model_across_stages['model_name']} from Stage '{best_overall_model_across_stages['stage']}' (CV RMSE: {-best_overall_model_across_stages['cv_score']:.4f})\n" # Print positive RMSE
        f"  Impact: Adding new data and preprocessing steps can lead to improvements in model performance by providing more diverse information or reducing noise."
    )
    stage_summaries.append(current_stage_summary)

    print(f"======== Finished Evaluation for Stage: {stage_name} ========\n")

# Convert compiled metrics to a DataFrame for easy viewing
metrics_df = pd.DataFrame(overall_metrics_results)
print("\n--- All Compiled Metrics ---")
print(metrics_df.to_string()) # Use to_string() to print full DataFrame without truncation

# Display a sample of the generated submission files
print("\n--- Sample of Generated Submission Files ---")
for key, df in overall_submission_dfs.items():
    print(f"\nSubmission: {key} (first 5 rows)")
    print(df.head())
    # You can save these to CSV files here:
    # df.to_csv(f'{key}.csv', index=False)

# Save the final best overall model
if best_overall_model_across_stages['model'] is not None:
    model_save_path = f'best_overall_multioutput_model_{datetime.now().strftime("%Y%m%d_%H%M%S")}.joblib'
    joblib.dump(best_overall_model_across_stages['model'], model_save_path)
    print(f"\nSaved overall best model to: {model_save_path}")
    # Print positive RMSE for readability
    print(f"  Model: {best_overall_model_across_stages['model_name']} from Stage '{best_overall_model_across_stages['stage']}' (CV RMSE: {-best_overall_model_across_stages['cv_score']:.4f})")
else:
    print("\nNo best overall model found to save.")

# Print the full summary of each stage's improvements
print("\n\n--- Full Summary of Each Stage's Improvements ---")
for summary_text in stage_summaries:
    print(summary_text)
    print("-" * 80) # Separator for readability



======== Starting Evaluation for Stage: Original Data ========

--- Processing Data for Stage: Original Data ---
Imputed missing values for: ['ecec20', 'hp20', 'xhp20', 'BulkDensity']
Scaled 'bio1' and 'bio7'.
Dropped 'xhp20'.
Dropped 'site' and 'PID' (from features).
Skipping advanced feature engineering and PCA as per request.
Dropped high VIF features: ['lon', 'lat', 'pH', 'alb', 'bio1', 'bio7', 'cec20', 'lstd', 'lstn', 'mb1', 'mb2', 'mb3', 'mb7', 'mdem', 'ph20', 'BulkDensity']
Final scaling applied. Train shape: (7744, 14), Test shape: (2418, 14)

Training and tuning multi-output regression models...
  Tuning and training XGBoost...


KeyboardInterrupt: 